In [ ]:
from google.colab import files
import pandas as pd
from openpyxl import load_workbook

import re
from collections import Counter
import requests
from bs4 import BeautifulSoup

#Proyecto : Identificar las palabras mas repetidas para el estudio de vocabulario en ingles.

##Tareas a desarrollar:
- Webscrapping para obtener el transcript de los links entregados en archivo excel.
- Obtener las palabras mas repetidas, considerando los phrasal verbs que son de especial interes para el solicitante
- Entregar un listado limpio de las palabras mas frecuentes para que posteriormente se identifique en que audios/transcript se pueden encontrar. El listado debe incluir la frecuencia de uso de la palabra.

In [ ]:

def get_transcript (url):
    """ Funcion para obtener el texto que referencia el transcript desde el excel. ejemplo de las urls que visita es la siguiente :
    url = 'https://speakenglishpodcast.com/374-the-90-rule-how-to-choose-english-listening-level/'
    """

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9" }

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    #se debe obtener el titulo de la pagina/audio y el cuerpo de transcript
    title = soup.find("h1", class_="entry-title").get_text(strip=True)
    transcript_section = soup.find( ["h1", "h2"], string=lambda x: x and "transcript" in x.lower() )

    transcript = ""
    if transcript_section:
        # tomar todos los siguientes parrafos
        for sibling in transcript_section.find_next_siblings():
            if sibling.name == "h2":
                break
            transcript += sibling.get_text(" ", strip=True) + " "

    # Cortar antes de "(END OF THE EXTRACT)"
    cut_text = "END OF THE EXTRACT)"
    if cut_text in transcript:
        transcript = transcript.split(cut_text)[0]

    return [title, transcript ]


### En el excel "link.xlsx" estan todos los hipervinculos que se deben buscar

In [ ]:
uploaded = files.upload()

Saving link.xlsx to link.xlsx


In [ ]:
#trabajar con libro para obtener el link
wb = load_workbook("link.xlsx")
ws = wb.active

#crear el df que voy a llenar
df_transcript=pd.DataFrame(columns=["title" , "transcript"])

for i in range(1,49):
  cell = ws.cell(row=i, column=1)
  url = cell.hyperlink.target
  df_transcript.loc[len(df_transcript)] = get_transcript (url) #append al dataframe

### Ejemplos

In [ ]:
df_transcript.iloc[3,1]

"Hi everyone! I'm Georgiana; founder of Speak English Podcast.com . My mission is to help YOU speak English fluently. In this episode: I’m going to talk about the relationship between reading , writing , speaking and listening. After that, I’m going to tell you a Point of View Story. Let’s get started! As a language student, the main activities to learn a new language are : reading, writing, listening, and speaking . This is what we naturally do in our mother tongue. One key aspect to keep in mind is that we can categorize these activities as input and output . As you may guess, listening and reading are input activities, and writing and speaking are output activities. In other words, when you’re listening or reading, you are being exposed to the language, and when you’re writing and speaking, you are “producing” the language. The traditional approach tells you that the more you write and speak, the better. That’s why language schools insist on writing a lot and “practicing” your speak

In [ ]:
df_transcript.iloc[10,1]

"Hi everyone! I'm Georgiana; founder of Speak English Podcast.com . My mission is to help YOU speak English fluently. In this episode: I'll be talking about passive listening, or in other words, listening in the background. After that, we’ll practice conversation skills with the powerful Question & Answer technique. Ok, let’s get started! In the language industry, there’s an appealing approach called passive listening . This means playing some English in the background while performing other tasks. In theory, you're learning because the brain is always learning, no matter how. Interesting, huh? You just need to play some English while you're cleaning, writing, surfing the Internet, etc. Well, although this approach may be appealing to many of you, here’s the bad news: It doesn’t work as a whole system for developing a complete fluency in English. The main drawback is that when we need to learn new content, our brain needs to pay attention, to be active. So, in an ideal world, we would 

In [ ]:
"""Codigo para guardar """

#df_transcript.to_csv("df_transcript.csv", index=False)
#files.download("df_transcript.csv")

"""codigo para cargar el avance"""
#uploaded = files.upload()
#df_transcript= pd.read_csv("df_transcript.csv")
#df_transcript


' codigo para cargar el avance'

## NLP process

In [ ]:
import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import spacy

In [ ]:
#prep para limpiar stop words y lemmitizar (is that a word?)

stop_words = stopwords.words('english')
stop_words.extend(['pdf', 'georgiana' , 'hi'])
stop_words = set(stop_words)

In [ ]:



#stop_words.extend(["pdf", "giorgiana"])
lemmatizer = WordNetLemmatizer()


def lemmatize_verb(word):
    return lemmatizer.lemmatize(word, 'v')

def lemmatize_word(word):
    return lemmatizer.lemmatize(word)

In [ ]:
#modelo en ingles / es_ español
nlp = spacy.load("en_core_web_sm")

def extract_words_spacy(text):

    if not isinstance(text, str):
        return []

    doc = nlp(text)

    results = []
    used_tokens = set()

    # Detectar phrasal verbs  con dependencias
    for token in doc:
        if token.dep_ == "prt":
            verb = token.head.text.lower()
            particle = token.text.lower()

            lemma = lemmatize_verb(verb) #eliminar ing, ed, etc
            pv = f"{lemma} {particle}"

            results.append(pv)

            # marcar verbo + particle como usados para no repetir el verbo
            used_tokens.add(token.i)
            used_tokens.add(token.head.i)

    # anexar palabras normales
    for token in doc:
        if token.i in used_tokens:
            continue

        word = token.text.lower()

        if word.isalpha() and word not in stop_words and token.pos_ != "PROPN":
            results.append(token.lemma_.lower())


    return list(set(results))

In [ ]:
df_transcript['words_spacy'] = df_transcript.iloc[:, 1].apply(extract_words_spacy)
df_transcript.head()

,title,transcript,words_spacy
0,#002 American and British accents,"Hi, everyone! I'm Georgiana, founder of SpeakE...","[soldier, move, new, film, start, sound, histo..."
1,#003 The Importance Of Repeated Listening,"Hello, everybody! I am Georgiana, your English...","[learn, move, seem, rule, new, hear, story, st..."
2,#004 Toastmasters- Speak English in your city,"Hi everyone! I'm Georgiana, founder of Speak E...","[learn, seem, story, start, sound, city, good,..."
3,#005 How to Speak English Fluently without gra...,Hi everyone! I'm Georgiana; founder of Speak E...,"[learn, output, episode, seem, point out, cont..."
4,#006 English Phrasal Verbs with mini-stories,Hi everyone! I'm Georgiana; founder of Speak E...,"[learn, episode, seem, need, story, start, eng..."


### Contar frecuencia

- contar las veces que aparece la palabra y listar en los audios que la palabra aparece

In [ ]:
from collections import Counter

# unir todas las listas en una sola
all_words = []

for words_list in df_transcript['words_spacy']:
    all_words.extend(words_list)

# contar frecuencia y ordenarla en df
freq = Counter(all_words)
d_frew = dict(freq)
dt_freq = pd.DataFrame(data= sorted(d_frew.items(), key=lambda item: item[0]) , columns= ["word" , "freq"])



In [ ]:
def find_rows(word, df):
    """ recibe la palabra y recorre el df para encontrar los audios en donde es mencionada"""
    rows = []

    for idx, words_list in enumerate(df["words_spacy"]):
        if word in words_list:
            rows.append(df.iloc[idx, 0])

    return rows

In [ ]:
dt_freq["podcast"] = dt_freq["word"].apply( lambda x: find_rows(x, df_transcript))
dt_freq.head()

,word,freq,podcast
0,able,3,"[#008 GET Practice Phrasal Verbs with a Story,..."
1,abroad,1,[#019 Is English Changing You?]
2,absorb,1,[#031 Have fun learning English with riddles]
3,academic,1,[#023 English Fluency – Mistakes when you spea...
4,accent,3,"[#002 American and British accents, #029 Under..."


In [ ]:
dt_freq.to_csv("dt_freq.csv", index=False)

files.download("dt_freq.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>